# Document Question Answering System (RAG)

## Objective

Develop a Retrieval-Augmented Generation (RAG) system to answer questions from custom documents by building a pipeline that retrieves relevant information from a document and uses a language model to generate answers covering:

1. **Document Ingestion** — load and extract text from PDF
2. **Text Chunking** — split raw text into overlapping chunks
3. **Embedding Generation** — encode chunks with all-MiniLM-L6-v2
4. **Vector Store** — build index and store embeddings
5. **Query Pipeline** — embed user question and retrieve top-k chunks
6. **Generation Model** — Answer generation model is used
7. **Answer Generation** — Flan-T5-Large generates a grounded answer
8. **Validation Logs** — run 5 sample questions and log results
9. **Optimisation Experiments** — chunk size tuning + hybrid search
10. **Final Demo** — final end-to-end demo
10. **System Metrics Report** — summary table of all design choices

The primary objective is to develop a simple Retrieval-Augmented Generation (RAG) system to answer questions from custom documents. Build a pipeline that retrieves relevant information from a document and uses a language model to generate answers.

*Importing Libraries*

In [75]:
import pdfplumber                        
import re                                 
import numpy as np                       
import faiss                             
import torch                              

from sentence_transformers import SentenceTransformer   
from transformers import T5Tokenizer, T5ForConditionalGeneration  
from rank_bm25 import BM25Okapi               

PDF_PATH          = "Resources/OS_unit1.pdf"
EMBEDDING_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL  = "google/flan-t5-large"

print("Libraries imported successfully")

Libraries imported successfully


**For the dataset a pdf containing notes of Operating System subject unit 1 is used**

## Step 1 — Document Ingestion

Getting the document's raw text into Python as a plain string.

We also do light cleaning — removing excessive whitespace and page-break artefacts —
because noisy text produces noisy embeddings, which hurts retrieval quality.


In [76]:
def load_pdf(path: str) -> str:
    all_text = []

    with pdfplumber.open(path) as pdf:
        print(f"Opened PDF: {path}")
        print(f"Total pages : {len(pdf.pages)}")

        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()
            if text:
                all_text.append(text)
                print(f"Page {page_num:>3}: {len(text):>5} characters extracted")
            else:
                print(f"Page {page_num:>3}: [no extractable text]")

    raw_text = "\n".join(all_text)
    return raw_text


def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


raw_text    = load_pdf(PDF_PATH)
clean       = clean_text(raw_text)

print(f"\nIngestion complete")
print(f"Raw text length: {len(raw_text):,} characters")
print(f"Cleaned text length: {len(clean):,} characters")


Opened PDF: Resources/OS_unit1.pdf
Total pages : 57
Page   1:  1225 characters extracted
Page   2:   546 characters extracted
Page   3:  1702 characters extracted
Page   4:  1938 characters extracted
Page   5:  1110 characters extracted
Page   6:  1098 characters extracted
Page   7:   992 characters extracted
Page   8:  1883 characters extracted
Page   9:   964 characters extracted
Page  10:  1818 characters extracted
Page  11:  1116 characters extracted
Page  12:  1133 characters extracted
Page  13:  1322 characters extracted
Page  14:   506 characters extracted
Page  15:   837 characters extracted
Page  16:   662 characters extracted
Page  17:   962 characters extracted
Page  18:   877 characters extracted
Page  19:  1204 characters extracted
Page  20:  1324 characters extracted
Page  21:   952 characters extracted
Page  22:   600 characters extracted
Page  23:  1286 characters extracted
Page  24:  2051 characters extracted
Page  25:   787 characters extracted
Page  26:  1402 charact

**we got clean string that will be used for creating chunks**

## Step 2 — Text Chunking

We split the text into **overlapping fixed-size chunks**.

A 50-character overlap ensures that boundary sentences appear fully in at least one chunk,
so no information is lost at the seam.


In [77]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    chunks = []
    start  = 0

    while start < len(text):
        end = start + chunk_size         
        chunk = text[start:end].strip()    

        if chunk:                          
            chunks.append(chunk)

        start += chunk_size - overlap      

    return chunks

chunks = chunk_text(clean, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

print(f"Chunking complete")
print(f"Chunk size: {CHUNK_SIZE} characters")
print(f"Overlap: {CHUNK_OVERLAP} characters")
print(f"Total chunks: {len(chunks)}")
print("\nSample Chunk 1")
print(chunks[0])
print()
if len(chunks) > 1:
    print("\nSample Chunk 2")
    print(chunks[1])

Chunking complete
Chunk size: 500 characters
Overlap: 50 characters
Total chunks: 139

Sample Chunk 1
Operating System An Operating System (OS) is an interface between a computer user and computer hardware. An operating system is software which performs all the basic tasks like file management, memory management, process management, handling input and output, and controlling peripheral devices such as disk drives and printers. An operating system is software that enables applications to interact with a computer's hardware. The software that contains the core components of the operating system is


Sample Chunk 2
ins the core components of the operating system is called the kernel. The primary purposes of an Operating System are to enable applications (spftwares) to interact with a computer's hardware and to manage a system's hardware and software resources. Some popular Operating Systems include Linux Operating System, Windows Operating System, VMS, OS/400, AIX, z/OS, etc. Operating sy

**The total chunk count tells us how many vectors we will store. Here 139 vectors will be stored**

## Step 3 — Generating Embeddings

We embed every chunk once and store the result. At query time we embed only the question —
a single fast operation.


In [78]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"Embedding model loaded: {EMBEDDING_MODEL}")
print(f"Output dimension: {embedder.get_embedding_dimension()}")

print(f"\nEmbedding {len(chunks)} chunks")

chunk_embeddings = embedder.encode(
    chunks,
    batch_size=32,          
    show_progress_bar=True,
    convert_to_numpy=True   
)

print(f"\nEmbedding complete")
print(f"Embedding matrix shape : {chunk_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9358.23it/s]


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2
Output dimension: 384

Embedding 139 chunks


Batches: 100%|██████████| 5/5 [00:01<00:00,  4.14it/s]


Embedding complete
Embedding matrix shape : (139, 384)


### Observation
The result is a 2D numpy array of shape (139, 384). Each row is the
semantic fingerprint of one text chunk. Similar chunks will have rows that
point in similar directions.


## Step 4 — Building the Vector Database

**I have used FAISS vector because:**
- Runs entirely locally
- Fast enough for document-scale RAG
- The reference GitHub repo used Pinecone, I have used FAISS to stay fully local


In [79]:
def build_vector_index(embeddings: np.ndarray) -> faiss.Index:
    dim = embeddings.shape[1]

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalised = embeddings / (norms + 1e-10)

    index = faiss.IndexFlatIP(dim)
    index.add(normalised.astype(np.float32))  

    return index, normalised

faiss_index, norm_embeddings = build_vector_index(chunk_embeddings)

print(f"FAISS index built")
print(f"Embedding dim: {faiss_index.d}")
print(f"Vectors stored: {faiss_index.ntotal}")
print(f"Search mode: Exact nearest-neighbour")

FAISS index built
Embedding dim: 384
Vectors stored: 139
Search mode: Exact nearest-neighbour


**The FAISS index now holds all chunk embeddings, normalised and ready for cosine
similarity search.**

## Step 5 — Query Embedding and Context Retrieval

This is the core of RAG: instead of relying on the LLM's training memory, we look up the answer
in our own document. The LLM only needs to do the final step: read the context and write a sentence.


In [80]:
TOP_K = 3        
def retrieve_chunks(query: str, index: faiss.Index,
                    chunks: list, embedder: SentenceTransformer,
                    top_k: int = TOP_K) -> list:
   
    query_vec = embedder.encode([query], convert_to_numpy=True)         
  
    query_vec = query_vec / (np.linalg.norm(query_vec, axis=1, keepdims=True) + 1e-10)
    query_vec = query_vec.astype(np.float32)

    scores, indices = index.search(query_vec, top_k)   

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:
            results.append({
                "chunk": chunks[idx],
                "score": float(score),
                "chunk_index": int(idx)
            })

    return results


test_query   = "What is an Operating system?"
test_results = retrieve_chunks(test_query, faiss_index, chunks, embedder, top_k=TOP_K)

print(f'Query: "{test_query}"')
print(f"Top-{TOP_K} retrieved chunks:\n")

for rank, r in enumerate(test_results, start=1):
    print(f"  Rank {rank} | Score: {r['score']:.4f} | Chunk #{r['chunk_index']}")
    print(f"  {r['chunk'][:200]}...")
    print()


Query: "What is an Operating system?"
Top-3 retrieved chunks:

  Rank 1 | Score: 0.9069 | Chunk #0
  Operating System An Operating System (OS) is an interface between a computer user and computer hardware. An operating system is software which performs all the basic tasks like file management, memory...

  Rank 2 | Score: 0.8087 | Chunk #31
  ating system.  Complex designing - Each virtual component of the machine is to be planned carefully as each component is to the abstract underlying hardware. Types of Operating Systems (OS) An operat...

  Rank 3 | Score: 0.7748 | Chunk #2
  ier • Make the computer system convenient to use • Use the computer hardware in an efficient manner Definitions An operating system is a program that acts as an interface between the user and the comp...



### Observation
The retrieval is working correctly when the returned chunks are semantically related
to the query — even if they do not share exact keywords.


## Step 6 — Loading the Generation Model

Here I have used Flan-T5-Large generation model. It is a Google instruction-following language model via Hugging-Face that can read a passage and answer questions about it — without an internet connection or API key.

In [81]:
tokenizer = T5Tokenizer.from_pretrained(GENERATION_MODEL)
gen_model  = T5ForConditionalGeneration.from_pretrained(
    GENERATION_MODEL,
    torch_dtype=torch.float32   
)

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_model = gen_model.to(device)
gen_model.eval() 

print(f"Generation model loaded")
print(f"Model: {GENERATION_MODEL}")
print(f"Device: {device.upper()}")
print(f"Mode: Inference (eval)")

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 13950.26it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation model loaded
Model: google/flan-t5-large
Device: CPU
Mode: Inference (eval)


**The model is now loaded and in eval mode. It disables dropout layers that are only needed during training, making inference deterministic and slightly faster.**


## Step 7 — Answer Generation with Grounded Context

We build a structured prompt that contains:
1. The retrieved context chunks
2. The user's question

The prompt explicitly tells the model to answer using only the context. The model cannot make up an
answer if we constrain it to the provided context.

In [82]:
def generate_answer(query: str,
                    context_chunks: list,
                    tokenizer,
                    model,
                    max_new_tokens: int = 256) -> str:
    
    context_parts = []
    
    for i, chunk_dict in enumerate(context_chunks, start=1):
        context_parts.append(f"[Passage {i}]: {chunk_dict['chunk']}")

    context_text = "\n".join(context_parts)

    prompt = (
        f"Answer the question using ONLY the context provided below. "
        f"Do not use any outside knowledge. If the answer is not in the context, "
        f"say 'The document does not contain enough information to answer this.'\n"
        f"Context:\n{context_text}\n"
        f"Question: {query}\n"
        f"Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",    
        max_length=1024,        
        truncation=True         
    ).to(device)


    with torch.no_grad():    
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,           
            early_stopping=True,    
            no_repeat_ngram_size=3 
        )

    
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return answer


def rag_pipeline(query: str) -> dict:
  
    retrieved = retrieve_chunks(query, faiss_index, chunks, embedder, top_k=TOP_K)
    answer = generate_answer(query, retrieved, tokenizer, gen_model)

    return {
        "query"    : query,
        "context"  : retrieved,
        "answer"   : answer
    }

result = rag_pipeline("What is the first layer in layered structure of an operating system")

print(f"QUERY: {result['query']}")
print("\nRETRIEVED CONTEXT:")

for i, r in enumerate(result["context"], 1):
    print(f"[{i}] Score: {r['score']:.4f} | {r['chunk'][:150]}...")
    
print(f"\nANSWER: {result['answer']}")


QUERY: What is the first layer in layered structure of an operating system

RETRIEVED CONTEXT:
[1] Score: 0.7889 | he bottom layer. All the layers hide some structures, operations, etc from their upper layers. One problem with the layered structure is that each lay...
[2] Score: 0.7232 | ly written code can ruin the system.  Complex Design - Exo-Kernel designing is complicated. 8 | P a ge OPERATING SYSTEM/5AID4-03/UNIT-1/ LECTURE NOTE...
[3] Score: 0.6787 | sily. A new functionality can be added without impacting other modules as well.  Verifiable - Being modular, each layer can be verified and debugged ...

ANSWER: The bottom layer


### Observation
**The answer is derived from the retrieved passages, not from the
model's training memory. If we look at the retrieved chunks, we are able to
trace exactly where the answer came from.**

## Step 8 — Validation Logs

We run 5 diverse questions and log:
- The question
- Each retrieved chunk with its similarity score
- The generated answer

This proves the pipeline works end-to-end and that answers are grounded in the document.


In [83]:
TEST_QUESTIONS = [
    "What is this document about?",
    "What is the full form of OS?",
    "What is the first layer in layered structure of an operating system",
    "What is the last layer of OS?",
    "What problems OS solves?"
]

validation_results = []

for q_num, question in enumerate(TEST_QUESTIONS, start=1):
    print(f"\nQuestion {q_num}/{len(TEST_QUESTIONS)}")
    print(f"QUERY: {question}")
    print()

    result = rag_pipeline(question)
    validation_results.append(result)
    
    print("RETRIEVED CONTEXT:")
    for rank, r in enumerate(result["context"], start=1):
        print(f"Rank {rank} | Similarity Score: {r['score']:.4f} | Chunk #{r['chunk_index']}")
        print(f'"{r['chunk'][:200]}"')
        print()

    print(f"GENERATED ANSWER:")
    print(f"{result['answer']}")
    
print("\nValidation complete — all 5 questions answered with context.")



Question 1/5
QUERY: What is this document about?

RETRIEVED CONTEXT:
Rank 1 | Similarity Score: 0.2919 | Chunk #122
"dcount variable denotes the number of readers accessing the file concurrently. The moment variable readcount becomes 1, wait operation is used to write semaphore which decreases the value by one. This"

Rank 2 | Similarity Score: 0.2639 | Chunk #85
"e entry Section decides the entry of a process. • Critical Section: The Critical section allows and makes sure that only one process is modifying the shared data. • Exit Section: The entry of other pr"

Rank 3 | Similarity Score: 0.2513 | Chunk #123
"for Writer Process 51 | P a ge OPERATING SYSTEM/5AID4-03/UNIT-1/ LECTURE NOTES Thread A thread is a flow of execution through the process code, with its own program counter that keeps track of which i"

GENERATED ANSWER:
OPERATING SYSTEM

Question 2/5
QUERY: What is the full form of OS?

RETRIEVED CONTEXT:
Rank 1 | Similarity Score: 0.7158 | Chunk #0
"Operating System An Operati

### Observation
**For each question, the retrieved chunks are semantically relevant to the question. The generated answers are consistent with what appears in the retrieved passages.**

## Step 9 — Optimisation: Chunk Size Tuning

We test three sizes on the same question and compare retrieval scores — a quantitative
way to pick the best chunk size for our specific document..


In [84]:
CHUNK_SIZES = [200, 500, 800]
EVAL_QUERY = "What is the full form of OS?"

print(f' Query: "{EVAL_QUERY}"\n')
print(f"{'Chunk Size':>12} | {'Num Chunks':>10} | {'Top-1 Score':>11} | {'Avg Top-3 Score':>15}")

chunk_size_results = {}

for cs in CHUNK_SIZES:
    exp_chunks = chunk_text(clean, chunk_size=cs, overlap=cs // 10)

    exp_embeddings = embedder.encode(exp_chunks, convert_to_numpy=True, show_progress_bar=False)

    exp_index, _ = build_vector_index(exp_embeddings)

    exp_results = retrieve_chunks(EVAL_QUERY, exp_index, exp_chunks, embedder, top_k=3)
    top1_score  = exp_results[0]["score"] if exp_results else 0.0
    avg_score   = np.mean([r["score"] for r in exp_results]) if exp_results else 0.0

    chunk_size_results[cs] = {
        "num_chunks": len(exp_chunks),
        "top1_score": top1_score,
        "avg_score": avg_score,
        "top_chunk": exp_results[0]["chunk"][:150] if exp_results else ""
    }

    print(f"{cs:>12} | {len(exp_chunks):>10} | {top1_score:>11.4f} | {avg_score:>15.4f}")

 Query: "What is the full form of OS?"

  Chunk Size | Num Chunks | Top-1 Score | Avg Top-3 Score
         200 |        347 |      0.7365 |          0.7127
         500 |        139 |      0.7158 |          0.6792
         800 |         87 |      0.7102 |          0.6536


### Observation
The chunk size with the highest Top-1 and Average Top-3 similarity score i.e. chunk size = 200 is the best
choice for our document.


## Optimisation: Hybrid Search

**Hybrid search** combines both:
- **BM25 score** — classic information retrieval, great for exact keyword/phrase matching
- **Vector score** — semantic similarity, great for paraphrase and meaning matching

In [85]:
HYBRID_ALPHA = 0.6 

def build_bm25_index(chunks: list) -> BM25Okapi:
    tokenized_chunks = [chunk.lower().split() for chunk in chunks]
    return BM25Okapi(tokenized_chunks)


def hybrid_retrieve(query: str,
                    faiss_index: faiss.Index,
                    bm25_index: BM25Okapi,
                    chunks: list,
                    embedder: SentenceTransformer,
                    top_k: int = TOP_K,
                    alpha: float = HYBRID_ALPHA) -> list:
    
    n = len(chunks)

    query_vec = embedder.encode([query], convert_to_numpy=True)
    query_vec = query_vec / (np.linalg.norm(query_vec, axis=1, keepdims=True) + 1e-10)
    query_vec = query_vec.astype(np.float32)

    vec_scores_raw, vec_indices = faiss_index.search(query_vec, n)
    vec_scores_raw = vec_scores_raw[0]  
    vec_indices    = vec_indices[0]       

    vec_score_map = np.zeros(n)
    for score, idx in zip(vec_scores_raw, vec_indices):
        if idx != -1:
            vec_score_map[idx] = score

    tokenized_query = query.lower().split()
    bm25_scores_raw = np.array(bm25_index.get_scores(tokenized_query))  

    def minmax_norm(arr):
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-10:
            return np.zeros_like(arr)
        return (arr - mn) / (mx - mn)

    vec_norm  = minmax_norm(vec_score_map)
    bm25_norm = minmax_norm(bm25_scores_raw)

    hybrid_scores = alpha * vec_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(hybrid_scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": float(hybrid_scores[idx]),
            "vec_score": float(vec_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_index": int(idx)
        })
    return results


bm25_index = build_bm25_index(chunks)

HYBRID_QUERY = "What is the full form of OS?"
print(f'Query: "{HYBRID_QUERY}"\n')

vec_results    = retrieve_chunks(HYBRID_QUERY, faiss_index, chunks, embedder, top_k=TOP_K)
hybrid_results = hybrid_retrieve(HYBRID_QUERY, faiss_index, bm25_index, chunks, embedder,
                                  top_k=TOP_K, alpha=HYBRID_ALPHA)

print(f"PURE VECTOR SEARCH (Top-{TOP_K})")
for i, r in enumerate(vec_results, 1):
    print(f"  [{i}] Score: {r['score']:.4f} | {r['chunk'][:120]}")

print(f"\n HYBRID SEARCH (Top-{TOP_K})")
for i, r in enumerate(hybrid_results, 1):
    print(f"  [{i}] Hybrid: {r['score']:.4f} (Vec:{r['vec_score']:.4f} | BM25:{r['bm25_score']:.4f}) | {r['chunk'][:120]}")

Query: "What is the full form of OS?"

PURE VECTOR SEARCH (Top-3)
  [1] Score: 0.7158 | Operating System An Operating System (OS) is an interface between a computer user and computer hardware. An operating sy
  [2] Score: 0.6743 | ating system.  Complex designing - Each virtual component of the machine is to be planned carefully as each component i
  [3] Score: 0.6475 | ins the core components of the operating system is called the kernel. The primary purposes of an Operating System are to

 HYBRID SEARCH (Top-3)
  [1] Hybrid: 0.8238 (Vec:0.9445 | BM25:0.6428) | ating system.  Complex designing - Each virtual component of the machine is to be planned carefully as each component i
  [2] Hybrid: 0.8081 (Vec:1.0000 | BM25:0.5203) | Operating System An Operating System (OS) is an interface between a computer user and computer hardware. An operating sy
  [3] Hybrid: 0.7389 (Vec:0.9085 | BM25:0.4843) | ins the core components of the operating system is called the kernel. The primary purpose

**Hybrid search gives better results than pure vector search**

## Step 10 — Final End-to-End Demo with Hybrid Retrieval

We run the complete pipeline one final time using hybrid retrieval — demonstrating
the optimised system in action.


In [86]:
def rag_pipeline_hybrid(query: str) -> dict:
    retrieved = hybrid_retrieve(query, faiss_index, bm25_index, chunks, embedder,
                                 top_k=TOP_K, alpha=HYBRID_ALPHA)
    answer    = generate_answer(query, retrieved, tokenizer, gen_model)
    return {"query": query, "context": retrieved, "answer": answer}

for q_num, question in enumerate(TEST_QUESTIONS, start=1):
    result = rag_pipeline_hybrid(question)

    print(f"Q{q_num}: {result['query']}")
    print(f"Top chunk score (hybrid): {result['context'][0]['score']:.4f}")
    print(f"Answer: {result['answer']}")
    print()

Q1: What is this document about?
Top chunk score (hybrid): 0.7651
Answer: Process Synchronization

Q2: What is the full form of OS?
Top chunk score (hybrid): 0.8238
Answer: Operating System

Q3: What is the first layer in layered structure of an operating system
Top chunk score (hybrid): 0.9504
Answer: bottom layer

Q4: What is the last layer of OS?
Top chunk score (hybrid): 0.9826
Answer: user interface

Q5: What problems OS solves?
Top chunk score (hybrid): 0.7460
Answer: Make the computer system convenient to use



### Observation
The hybrid pipeline is the final, optimised version of the system.
It combines the semantic power of vector search with the precision of keyword matching.


## Step 11 — System Metrics Report

this is the system metrics report detailing chunking profiles, chosen
text embedding dimensions, vector store tools, and language model setups.


In [87]:
import datetime

num_chunks   = len(chunks)
avg_chunk_len = np.mean([len(c) for c in chunks])
embed_dim    = embedder.get_embedding_dimension()
total_vectors = faiss_index.ntotal
doc_pages    = len(pdfplumber.open(PDF_PATH).pages) if PDF_PATH.endswith(".pdf") else "N/A"

best_cs = max(chunk_size_results, key=lambda k: chunk_size_results[k]["avg_score"])

print("RAG SYSTEM METRICS REPORT")
print(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")

print()

print("DOCUMENT INGESTION")
print(f"Source type: PDF (own document)")
print(f"Total pages: {doc_pages}")
print(f"Raw text length: {len(raw_text):,} characters")
print(f"Cleaned text length: {len(clean):,} characters")

print()

print("CHUNKING PROFILE")
print(f"Strategy: Fixed-size with overlap")
print(f"Chunk size: {CHUNK_SIZE} characters")
print(f"Overlap: {CHUNK_OVERLAP} characters")
print(f"Total chunks: {num_chunks}")
print(f"Avg chunk length: {avg_chunk_len:.1f} characters")
print(f"Best chunk size: {best_cs} chars")

print()

print("EMBEDDING MODEL")
print(f"Model name: {EMBEDDING_MODEL}")
print(f"Embedding dimension: {embed_dim}")
print(f"Framework: sentence-transformers")
print(f"Total vectors: {total_vectors}")

print()

print("VECTOR STORE")
print(f"Library: FAISS")
print(f"Search mode: Exact nearest-neighbour")
print(f"Top-K retrieved: {TOP_K} chunks per query")

print()

print("RETRIEVAL STRATEGY")
print(f"Primary method: Hybrid search (vector + BM25 keyword)")
print(f"Alpha (vector wt.): {HYBRID_ALPHA} | BM25 weight: {1 - HYBRID_ALPHA}")
print(f"Normalisation: Min-max on both scores before weighted sum")

print()

print("GENERATION MODEL")
print(f"Model name: {GENERATION_MODEL}")
print(f"Framework: HuggingFace Transformers")
print(f"Device: {device.upper()}")
print(f"Decoding strategy: Beam search (num_beams=4, no_repeat_ngram_size=3)")
print(f"Max output tokens: 256")

print()

print("EXPERIMENT SUMMARY")

for cs, res in chunk_size_results.items():
    print(f"  Chunk size {cs:>4}: {res['num_chunks']:>4} chunks | Top-1 score: {res['top1_score']:.4f} | Avg-3 score: {res['avg_score']:.4f}")
print()

RAG SYSTEM METRICS REPORT
Generated: 2026-07-06 02:26

DOCUMENT INGESTION
Source type: PDF (own document)
Total pages: 57
Raw text length: 62,458 characters
Cleaned text length: 62,458 characters

CHUNKING PROFILE
Strategy: Fixed-size with overlap
Chunk size: 500 characters
Overlap: 50 characters
Total chunks: 139
Avg chunk length: 498.7 characters
Best chunk size: 200 chars

EMBEDDING MODEL
Model name: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384
Framework: sentence-transformers
Total vectors: 139

VECTOR STORE
Library: FAISS
Search mode: Exact nearest-neighbour
Top-K retrieved: 3 chunks per query

RETRIEVAL STRATEGY
Primary method: Hybrid search (vector + BM25 keyword)
Alpha (vector wt.): 0.6 | BM25 weight: 0.4
Normalisation: Min-max on both scores before weighted sum

GENERATION MODEL
Model name: google/flan-t5-large
Framework: HuggingFace Transformers
Device: CPU
Decoding strategy: Beam search (num_beams=4, no_repeat_ngram_size=3)
Max output tokens: 256

EXPERIME

### Observation
Every number in this report comes from the actual
live run — there is nothing hardcoded here. The chunk size experiment winner is
determined automatically by comparing retrieval scores.

## Key Learnings

1. **RAG solves the private-data problem** — a pre-trained LLM has zero knowledge of our PDF. RAG gives it that knowledge at query time without any retraining.

2. **The retrieval step is everything** — if retrieval fails, no LLM, however powerful, can generate a correct answer. Chunking strategy and embedding quality are more impactful than the generator model choice.

3. **Chunk size is a tunable hyperparameter** — not a fixed number. The right size depends on our document's structure.

4. **Hybrid search is strictly better than pure vector search** — it catches both semantic meaning and exact keyword matches. The only cost is slightly more complexity.


## Conclusion

This project successfully demonstrates a complete Retrieval-Augmented Generation pipeline built using open-source tools — covering every stage from document ingestion and chunking to vector search and local answer generation with Flan-T5-Large. The optimisation experiments confirmed that chunk size and hybrid BM25 + vector retrieval are meaningful design choices that directly impact answer quality, and the system runs entirely without any API key or cloud dependency.

Beyond the implementation, this assignment reinforced a fundamental principle of modern AI: a language model's internal knowledge is insufficient for reliable question answering over private or domain-specific data. RAG bridges that gap by grounding every generated answer in retrieved evidence, making the system transparent, verifiable, and practical — the same architecture that powers real-world enterprise search engines, documentation assistants, and intelligent chatbots.